In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from rag_helper import RAGBase

In [4]:
class RAGTraced(RAGBase):

    def rag(self, query):
        print("ENTER rag")
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        print("ENTER search")
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        print("ENTER llm")
        with tracer.start_as_current_span("llm"):
            return super().llm(prompt)

In [5]:
from starter import index, client
rag = RAGTraced(
    index=index,
    llm_client=client
)

In [6]:
import sys
sys.stdout.flush()

Q1. First trace

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

print(answer)

ENTER rag
ENTER search
{
    "name": "search",
    "context": {
        "trace_id": "0xd2a7eb72789ab161c87398bb6b54ae4c",
        "span_id": "0x9bfb5599a6ac5740",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0cb3e2f9289c35c1",
    "start_time": "2026-07-21T13:49:08.657537Z",
    "end_time": "2026-07-21T13:49:08.660065Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "cd1d980c-53b7-4e04-b058-9c30b67e85ba",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
AFTER BUILD PROMPT
ENTER llm
{
    "name": "llm",
    "context": {
        "trace_id": "0xd2a7eb72789ab161c87398bb6b54ae4c",
        "span_id": "0x82fb9e3a1fbc98a0",
    

In [8]:
class RAGTraced(RAGBase):

    def rag(self, query):
        print("ENTER rag")
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        print("ENTER search")
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute(
                "input_tokens",
                usage.prompt_tokens
            )

            span.set_attribute(
                "output_tokens",
                usage.completion_tokens
            )

            return response

In [9]:
rag = RAGTraced(
    index=index,
    llm_client=client
)

Q2. Capturing metrics as span attributes

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

print(answer)

ENTER rag
ENTER search
{
    "name": "search",
    "context": {
        "trace_id": "0x7cd050ab3c75824a150c5fb5c2141bbb",
        "span_id": "0xfd066ef84efef160",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x401f87d0af20c42a",
    "start_time": "2026-07-21T13:49:11.423263Z",
    "end_time": "2026-07-21T13:49:11.427640Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "cd1d980c-53b7-4e04-b058-9c30b67e85ba",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
AFTER BUILD PROMPT
{
    "name": "llm",
    "context": {
        "trace_id": "0x7cd050ab3c75824a150c5fb5c2141bbb",
        "span_id": "0xdaf8801aeb286ce6",
        "trace

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ktvezbv5fgcsn5xhvyxqbnz3` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99599, Requested 5743. Please try again in 1h16m55.488s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Q3. Span timing

start_time: 18.057894Z
end_time:   20.203887Z

In [ ]:
answer = 20.203887 - 18.057894
print(answer * 1000)

2145.993000000001


In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="my_traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        print(f"Exporting {len(spans)} span(s)")
        for span in spans:
            print(span.name)
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("my_traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from rag_helper import RAGBase
class RAGTraced(RAGBase):

    def rag(self, query):
        print("ENTER rag")
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query):
        print("ENTER search")
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute(
                "input_tokens",
                usage.prompt_tokens
            )

            span.set_attribute(
                "output_tokens",
                usage.completion_tokens
            )

            return response

In [5]:
from starter import index, client

rag = RAGTraced(
    index=index,
    llm_client=client
)

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)

ENTER rag
ENTER search
Exporting 1 span(s)
search
AFTER BUILD PROMPT


Exporting 1 span(s)
llm
AFTER LLM
Exporting 1 span(s)
rag


In [8]:
import sqlite3

conn = sqlite3.connect("my_traces.db")

print(
    conn.execute(
        "SELECT * FROM spans"
    ).fetchall()
)

[('search', 1784705163689853090, 1784705163692648249, None, None, None), ('llm', 1784705163700317572, 1784705166176809355, 5742, 554, None), ('rag', 1784705163689729209, 1784705166180879524, None, None, None)]


In [16]:
import os

print(os.getcwd())
print(os.path.exists("traces.db"))
print(os.path.getsize("traces.db"))

/workspaces/RAG_Zoomcamp/llm-zoomcamp-hw5
True
8192


In [15]:
print(os.path.abspath("traces.db"))

/workspaces/RAG_Zoomcamp/llm-zoomcamp-hw5/traces.db


In [17]:
conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

[('spans',)]